# HyDE를 이용한 질의 확장

HyDE(Hypothetical Document Embeddings, 가상 문서 임베딩)는 짧은 질문 대신 LLM이 작성한 가상 답변을 임베딩해 검색하는 질의 확장 기법이다. 질문을 실제 문서에 가까운 어휘와 문체의 검색 표현으로 바꾸면 Dense Retrieval이 관련 문서를 찾기 쉬워질 수 있다.

이 단원은 핵심 검색 경로를 익힌 뒤 필요에 따라 선택하는 독립 심화 분기이다. 실행에는 BM25·Dense 검색의 기본 개념, 원문 문서가 적재된 Pinecone 인덱스, OpenAI·Pinecone 설정이 필요하며 패키지와 CSV는 이 노트북에서 준비한다. 메타데이터 필터링·Self-Query·Rerank·문서 압축의 출력을 사용하지 않으며, 질문의 검색 표현이 병목일 때 HyDE를 선택한다.

전체 흐름은 `질문 → 가상 답변 → 문자열 → 임베딩 → Pinecone 검색 → 실제 Document`이다. 가상 답변은 검색 공간에서 문서를 찾기 위한 표현일 뿐 사실 근거나 최종 답변이 아니다. 근거는 검색으로 반환된 실제 `Document`의 본문과 메타데이터에서 확인해야 한다.

이 실습은 짧거나 추상적인 질문을 확장할 때의 장점과, 추가 LLM 비용·지연 및 잘못된 가정이 검색을 왜곡할 수 있다는 한계를 함께 다룬다. BM25·일반 Dense·HyDE를 같은 정답 집합과 순위 지표로 비교한다.


## HyDE 실행 패키지 준비

`%pip`은 현재 Jupyter 커널에 필요한 패키지를 설치한다. BM25·Dense 기준선, OpenAI 생성 모델과 Pinecone 검색 경로를 준비한다. 설치 후 커널이 이전 모듈을 잡고 있으면 한 번 재시작한다.

### 코드 해석 순서

1. HyDE 실습에 필요한 공식 패키지를 현재 커널에 설치한다.

### 결과 해석

- 패키지별 설치 로그가 나타나며 의존성 충돌이 없으면 셀이 종료된다.
- 설치가 끝나면 다음 셀에서 HyDE의 모델·검색기 객체를 직접 import할 수 있다.


In [1]:
# 설치 목록은 HyDE 활성 코드에 필요한 SDK와 분석 도구를 포함한다.
# `-U`는 이미 설치된 패키지를 호환되는 최신 배포본으로 갱신한다.
%pip install -U pandas numpy rank_bm25 konlpy langchain langchain-openai langchain-pinecone pinecone python-dotenv gdown tqdm


  Using cached pinecone-9.1.0-cp310-abi3-win_amd64.whl.metadata (6.3 kB)
Note: you may need to restart the kernel to use updated packages.


## HyDE 환경 변수와 모델 이름 준비

`load_dotenv()`는 `.env`에 저장한 OpenAI·Pinecone 설정을 현재 Python 환경으로 불러온다. API key는 출력하지 않으며 OpenAI와 Pinecone SDK가 환경 변수에서 직접 사용한다.

이 노트북에서 직접 사용하는 설정만 준비한다.

- `OPENAI_LLM_MODEL`: 가상 문서를 생성할 Chat Model이다.
- `PINECONE_INDEX_NAME`: 원문이 저장된 Pinecone index 이름이다.
- `OPENAI_EMBEDDING_MODEL`: 질문과 가상 문서를 벡터로 바꿀 임베딩 모델이다.

### 코드 해석 순서

1. `.env`를 불러오고 HyDE에서 직접 사용할 모델과 Pinecone index를 지정한다.

### 결과 해석

- 환경 변수와 실습용 모델 설정이 준비되며 화면에 별도 출력은 나타나지 않는다.
- HyDE는 생성 모델, 원문 index와 임베딩 모델 설정만 있으면 바로 실행할 수 있다.


In [2]:
import os
from dotenv import load_dotenv

# load_dotenv()는 `.env`의 값을 환경 변수에 추가하며 API key를 화면에 출력하지 않는다.
load_dotenv()

# 생성 모델은 뒤의 ChatOpenAI가 질문을 가상 문서로 바꿀 때 사용한다.
OPENAI_LLM_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")

# index 이름과 임베딩 모델은 뒤의 PineconeVectorStore와 OpenAIEmbeddings에 전달한다.
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "adv-rag")
OPENAI_EMBEDDING_MODEL = os.getenv(
    "OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"
)


## HyDE 비교 데이터 다운로드

문서 corpus와 질의·정답 CSV를 내려받는다. 같은 데이터에서 질문 임베딩과 가상 문서 임베딩을 비교해야 HyDE 변환의 효과를 분리할 수 있다.

### 코드 해석 순서

1. HyDE와 기준 검색기에 공통으로 사용할 두 CSV를 저장한다.

### 결과 해석

- 실행하면 문서와 질의 파일의 다운로드 완료 진행률이 나타난다.
- 고정된 질의·정답을 사용하면 생성 내용이 달라져도 검색 지표를 같은 기준으로 비교할 수 있다.


In [3]:
# 문서 파일은 Pinecone에 이미 적재된 ID와 본문을 확인하는 기준이다.
# 질의 파일은 가상 문서 생성 입력과 검색 평가용 qrels를 함께 제공한다.
# documents.csv
!gdown 1pspaw_q4_QCp2K4M-thDlUtra-_wQI7k
# queries.csv
!gdown 1dsn0pwkfzOUxiQ4MKDIvMM-CkxbYiSce


Downloading...
From: https://drive.google.com/uc?id=1pspaw_q4_QCp2K4M-thDlUtra-_wQI7k
To: C:\SKN_AI\09_llm\07_advanced_rag\01_retrieval_optimization\documents.csv

  0%|          | 0.00/14.9k [00:00<?, ?B/s]
100%|██████████| 14.9k/14.9k [00:00<?, ?B/s]
Downloading...
From: https://drive.google.com/uc?id=1dsn0pwkfzOUxiQ4MKDIvMM-CkxbYiSce
To: C:\SKN_AI\09_llm\07_advanced_rag\01_retrieval_optimization\queries.csv

  0%|          | 0.00/2.19k [00:00<?, ?B/s]
100%|██████████| 2.19k/2.19k [00:00<?, ?B/s]


## 문서·질의 DataFrame 준비

두 CSV를 DataFrame으로 읽고 `queries_df`를 표시한다. `query_text`는 BM25·Dense·HyDE의 공통 시작 입력이고, `relevant_doc_ids`는 검색 결과를 채점할 qrels(query relevance judgments)이다.

질의 ID인 `query_id`는 가상 답변, 세 검색 결과와 평가 정답을 연결하는 key로 계속 유지된다.

### 코드 해석 순서

1. documents_df는 BM25 corpus와 문서 ID 매핑에 사용된다.
2. queries_df의 각 행은 가상 답변 생성과 qrels 평가의 입력이 된다.

### 결과 해석

- 실행하면 Q1부터 Q30까지의 질문과 관련 문서 등급이 표시된다.
- 질의 ID가 가상 문서와 세 검색 결과의 공통 key로 유지되어야 평가가 어긋나지 않는다.


In [4]:
import pandas as pd

# 1. documents_df는 BM25 corpus와 문서 ID 매핑에 사용된다.
documents_df = pd.read_csv('documents.csv')

# 2. queries_df의 각 행은 가상 답변 생성과 qrels 평가의 입력이 된다.
queries_df = pd.read_csv('queries.csv')
queries_df


,query_id,query_text,relevant_doc_ids
0,Q1,제주도 올레길 트레킹 코스 추천,D1=3;D4=1;D30=1
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,D2=3
2,Q3,걸스데이 대표 히트곡 목록 알려줘,D3=3
3,Q4,훈민정음 창제 배경과 세종대왕의 의의,D4=3
4,Q5,이순신 장군이 명량 해전에서 사용한 전술은 무엇인가?,D5=3
5,Q6,2024년 기후 변화 주요 지표와 한국의 탄소 중립 정책,D6=3;D14=2;D26=1
6,Q7,한국 AI 윤리 이슈와 관련 정책 사례는?,D7=3;D25=2
7,Q8,서울 지하철 환승 시 T-money 사용 방법,D8=3
8,Q9,판소리 춘향가 줄거리와 공연 특징,D9=3
9,Q10,한국 축구 대표팀 2002년 한일 월드컵 4강 진출 이유,D10=3


## HyDE 기준선 BM25 구성

BM25는 질문과 문서를 형태소 단위로 비교하는 희소 검색 기준선이다. Okt로 문서와 질문을 같은 방식으로 토큰화하고, 점수가 높은 행의 `doc_id`를 순서대로 반환한다.

`bm25_search()`의 입력은 질문 문자열과 `top_k`이고, 출력은 `list[str]` 형태의 문서 ID 목록이다. 이 계약은 뒤에서 Dense·HyDE 결과와 같은 평가 함수에 연결된다.

### 코드 해석 순서

1. 문서 본문을 형태소 목록으로 바꿔 BM25 corpus를 만든다.
2. 질문 문자열을 같은 방식으로 토큰화하고 상위 문서 ID를 list[str]로 반환한다.

### 결과 해석

- BM25 객체와 함수가 정의되며 이 셀에는 별도 검색 결과가 출력되지 않는다.
- 동일 질문의 키워드 기준선을 보존하면 HyDE가 의미 표현을 바꾼 효과를 분리해 볼 수 있다.


In [5]:
from konlpy.tag import Okt
from rank_bm25 import BM25Okapi

# 1. 문서 본문을 형태소 목록으로 바꿔 BM25 corpus를 만든다.
okt = Okt()
tokenized_docs = [okt.morphs(content) for content in documents_df['content']]
bm25 = BM25Okapi(tokenized_docs)

# 2. 질문 문자열을 같은 방식으로 토큰화하고 상위 문서 ID를 list[str]로 반환한다.
def bm25_search(query, top_k=5):
    query_token = okt.morphs(query)
    scores = bm25.get_scores(query_token)
    sorted_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    # 점수 인덱스를 원래 DataFrame의 doc_id로 복원해 평가 함수에 전달한다.
    ranked_docs = [documents_df['doc_id'].iloc[i] for i in sorted_idx[:top_k]]
    return ranked_docs


## HyDE 기준선 Dense 검색 연결

`OpenAIEmbeddings`는 질문이나 가상 답변 문자열을 임베딩 벡터로 바꾼다. `model`은 문서를 색인할 때 사용한 임베딩 모델과 같아야 한다.

`PineconeVectorStore`의 `index_name`은 검색할 Pinecone index를 선택하고, `embedding`은 검색 입력을 벡터로 바꾸는 객체이다. 이 코드에는 `namespace` 인자가 없으므로 기본 namespace를 사용한다. 뒤의 `similarity_search()`는 일반 질문과 가상 답변을 받아 각각 `list[Document]`를 반환한다.

### 코드 해석 순서

1. model은 검색 입력을 문서 색인과 같은 벡터 공간으로 변환할 임베딩 모델이다.
2. index_name은 검색할 Pinecone index이고 embedding은 질문을 벡터로 바꾸는 객체이다.

### 결과 해석

- 두 클라이언트 객체가 생성되며 검색 결과는 뒤의 비교 셀에서 만든다.
- 뒤의 similarity_search()는 두 입력을 각각 list[Document]로 반환하므로 차이는 검색 입력 문자열에 있다.


In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 1. model은 검색 입력을 문서 색인과 같은 벡터 공간으로 변환할 임베딩 모델이다.
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 2. index_name은 검색할 Pinecone index이고 embedding은 질문을 벡터로 바꾸는 객체이다.
# namespace 인자를 전달하지 않으므로 similarity_search()는 기본 namespace를 사용한다.
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
)


## 질의별 가상 답변 생성

`hyde_chain`은 `PromptTemplate → ChatOpenAI → StrOutputParser` 순서로 연결된다. 입력 딕셔너리의 `query`가 프롬프트 문자열에 들어가고, `ChatOpenAI`가 `AIMessage`를 반환하며, `StrOutputParser`가 메시지의 내용을 평문 `str`로 꺼낸다.

`ChatOpenAI`의 `model`은 사용할 생성 모델, `use_responses_api`는 Responses API 경로, `temperature`는 표현의 변동성, `reasoning_effort`는 현재 호출의 추론 설정을 정한다. 최종 문자열인 `pseudo_answer`만 `hyde_pseudo[qid]`에 저장하며, 이 값은 다음 Pinecone 검색 입력으로만 사용하며 근거로 취급하지 않는다.

### 코드 해석 순서

1. 입력 딕셔너리의 query를 검색용 가상 답변 지시문에 삽입한다.
2. ChatOpenAI는 완성된 프롬프트를 받아 AIMessage를 반환한다.
3. StrOutputParser는 AIMessage.content를 평문 str로 바꿔 invoke()의 반환형을 맞춘다.
4. 질의 ID별 가상 답변 문자열을 저장해 다음 DataFrame과 Pinecone 검색에 연결한다.

### 결과 해석

- API 경로를 실행하면 진행률이 30건까지 증가하고 질의별 가상 문서 딕셔너리가 채워진다.
- 가상 문서 생성이 성공해도 사실성을 보장하지 않으며 실제 근거는 후속 검색의 `Document`에서 찾아야 한다.


In [ ]:
# `hyde_chain`의 입력 `{'query': str}`은 프롬프트와 LLM을 거쳐 하나의 문자열이 된다.
# `hyde_pseudo[qid]`는 최종 답변이 아니라 다음 Pinecone 검색의 입력으로만 사용한다.
from tqdm import tqdm

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1. 입력 딕셔너리의 query를 검색용 가상 답변 지시문에 삽입한다.
hyde_prompt = PromptTemplate.from_template("""
다음 질문과 관련된 실제 문서가 사용할 법한 표현으로 짧은 가상 문서를 작성한다.
가상 문서는 검색 입력일 뿐 사실 출처나 최종 답변이 아니다.

질문: {query}
가상 문서:
""")

# 2. ChatOpenAI는 완성된 프롬프트를 받아 AIMessage를 반환한다.
hyde_llm = ChatOpenAI(
    # model은 환경 변수로 선택한 생성 모델이다.
    model=OPENAI_LLM_MODEL,
    # use_responses_api=True는 Responses API 경로를 사용한다.
    use_responses_api=True,
    # temperature=0.3은 검색 표현의 변동성을 낮게 유지한다.
    temperature=0.3,
    # reasoning_effort='none'은 이 호출의 추론 설정을 지정한다.
    reasoning_effort="none",
)

# 3. StrOutputParser는 AIMessage.content를 평문 str로 바꿔 invoke()의 반환형을 맞춘다.
hyde_chain = hyde_prompt | hyde_llm | StrOutputParser()

# 4. 질의 ID별 가상 답변 문자열을 저장해 다음 DataFrame과 Pinecone 검색에 연결한다.
hyde_pseudo = {}
for _, row in tqdm(queries_df.iterrows(), total=len(queries_df)):
    qid = row["query_id"]
    query = row["query_text"]

    # invoke()는 query dict를 받아 AIMessage를 거친 최종 str을 반환한다.
    pseudo_answer = hyde_chain.invoke({"query": query})
    # 이 문자열은 답변으로 제시하지 않고 같은 qid의 검색 입력으로 보존한다.
    hyde_pseudo[qid] = pseudo_answer


 60%|██████    | 18/30 [00:50<00:31,  2.58s/it]

## 질문과 가상 답변 나란히 확인

`hyde_pseudo`는 `query_id → 가상 답변 문자열` 매핑이다. 이를 `queries_df`와 같은 질의 순서로 DataFrame에 배치하면 원문 질문과 검색용 표현을 한 행에서 비교할 수 있다.

`pseudo_answer` 열은 사실을 채점하는 답안이 아니다. 질문의 핵심 개념과 실제 문서에서 사용할 법한 표현을 포함하는지, 불필요한 가정이 검색 방향을 바꾸지는 않는지 확인한다.

### 코드 해석 순서

1. 긴 가상 답변 문자열을 줄이지 않고 확인하도록 표시 폭을 설정한다.
2. query_id 순서를 기준으로 질문과 검색용 가상 답변을 같은 행에 배치한다.

### 결과 해석

- 생성 경로를 실행하면 `query_id`, `query_text`, `pseudo_answer` 세 열의 표가 표시된다.
- 질문과 무관한 세부 사실이 반복되면 HyDE가 검색을 왜곡할 수 있으므로 프롬프트나 생성 온도를 조정해야 한다.


In [ ]:
# 1. 긴 가상 답변 문자열을 줄이지 않고 확인하도록 표시 폭을 설정한다.
pd.set_option("display.max_colwidth", None)

# 2. query_id 순서를 기준으로 질문과 검색용 가상 답변을 같은 행에 배치한다.
hyde_pseudo_df = pd.DataFrame({
    "query_id": queries_df["query_id"],
    "query_text": queries_df["query_text"],
    "pseudo_answer": [hyde_pseudo[qid] for qid in queries_df["query_id"]],
})

hyde_pseudo_df.head()


## HyDE 검색 평가 함수 정의

세 검색 결과를 같은 qrels와 같은 `k=5` 기준으로 평가한다. P@k는 상위 k 중 관련 문서 비율, R@k는 전체 관련 문서 중 상위 k가 찾은 비율, MRR은 첫 관련 문서 순위의 역수, MAP는 관련 문서가 나타날 때마다 계산한 정밀도의 평균이다.

평가 함수의 입력은 `query_id → list[doc_id]` 딕셔너리와 질의 DataFrame이다. AP@k의 분모에 상위 k에서 놓친 관련 문서도 반영해 순위 품질을 과대평가하지 않는다.

### 코드 해석 순서

1. qrels 문자열을 관련 문서 ID와 등급의 딕셔너리로 변환한다.
2. 예측 ID 순서와 정답 딕셔너리에서 P@k, R@k, RR와 AP@k를 계산한다.
3. 모든 질의의 네 지표를 평균내 검색 방식별 비교 딕셔너리를 반환한다.

### 결과 해석

- 평가 함수만 정의되며 직접 출력은 없고 마지막 표 생성에서 호출된다.
- HyDE 추가 비용을 정당화하려면 적어도 회수율이나 상위 순위 지표의 측정 개선이 필요하다.


In [ ]:
# 질의별 문서 ID 순서와 qrels 딕셔너리가 지표 함수의 두 핵심 입력이다.
# 네 지표의 질의 평균이 검색 방식별 비교 딕셔너리로 반환된다.
import numpy as np


# 1. qrels 문자열을 관련 문서 ID와 등급의 딕셔너리로 변환한다.
def parse_relevant(relevant_str):
    # 입력 예시 `D6=3;D14=2`를 문서 ID와 관련성 등급의 딕셔너리로 변환한다.
    relevant_dict = {}
    for pair in relevant_str.split(";"):
        doc_id, grade_text = pair.split("=")
        grade = int(grade_text)
        # 0등급 문서가 포함되더라도 정답 hit로 세지 않고 1 이상만 보존한다.
        if grade > 0:
            relevant_dict[doc_id] = grade
    return relevant_dict


# 2. 예측 ID 순서와 정답 딕셔너리에서 P@k, R@k, RR와 AP@k를 계산한다.
def compute_metrics(predicted, relevant_dict, k=5):
    # predicted[:k]가 평가 대상이며 관련성 등급이 1 이상인 문서를 정답으로 처리한다.
    top_k = predicted[:k]
    hits = sum(doc_id in relevant_dict for doc_id in top_k)
    precision = hits / k

    total_relevant = len(relevant_dict)
    recall = hits / total_relevant if total_relevant else 0.0

    # RR은 첫 관련 문서의 순위 역수이므로 첫 정답이 1위이면 1.0이다.
    rr = next(
        (1 / rank for rank, doc_id in enumerate(predicted, start=1) if doc_id in relevant_dict),
        0.0,
    )

    # AP@k는 관련 문서를 만난 각 순위의 Precision을 min(관련 문서 수, k)로 나눈다.
    precision_sum = 0.0
    relevant_seen = 0
    for rank, doc_id in enumerate(top_k, start=1):
        if doc_id in relevant_dict:
            relevant_seen += 1
            precision_sum += relevant_seen / rank
    ap_denominator = min(total_relevant, k)
    ap = precision_sum / ap_denominator if ap_denominator else 0.0
    return precision, recall, rr, ap


# 3. 모든 질의의 네 지표를 평균내 검색 방식별 비교 딕셔너리를 반환한다.
def evaluate_all(method_results, queries_df, k=5):
    # 각 질의의 네 지표를 누적한 뒤 평균을 반환해 검색기 간 비교표에 사용한다.
    per_query_metrics = []
    for _, row in queries_df.iterrows():
        relevant_dict = parse_relevant(row["relevant_doc_ids"])
        predicted = method_results[row["query_id"]]
        per_query_metrics.append(compute_metrics(predicted, relevant_dict, k))

    metric_array = np.asarray(per_query_metrics, dtype=float)
    return {
        "P@k": metric_array[:, 0].mean(),
        "R@k": metric_array[:, 1].mean(),
        "MRR": metric_array[:, 2].mean(),
        "MAP": metric_array[:, 3].mean(),
    }


## 세 검색 방식의 질의별 결과 생성

BM25는 원문 질문을 받아 문서 ID 목록을 직접 반환한다. Dense와 HyDE는 각각 원문 질문과 가상 답변을 `similarity_search()`에 전달해 `list[Document]`를 받고, 각 `Document.metadata['doc_id']`를 `list[str]`로 변환한다.

세 결과의 최종 계약은 모두 `dict[query_id, list[doc_id]]`이다. 따라서 같은 `evaluate_all()`에 전달하면 검색 입력을 바꾼 효과만 같은 기준으로 비교할 수 있다.

### 코드 해석 순서

1. BM25는 원문 질문을 받아 list[doc_id]를 직접 반환한다.
2. 일반 Dense는 원문 질문을 검색하고 list[Document]를 list[doc_id]로 바꾼다.
3. HyDE는 가상 답변을 검색하고 같은 list[doc_id] 계약으로 맞춘다.

### 결과 해석

- Pinecone 경로를 실행하면 세 질의별 결과 딕셔너리가 채워지며 이 셀은 별도 표를 출력하지 않는다.
- 출력 형식을 통일하면 검색 입력 변환만 다른 세 방법을 동일 지표로 비교할 수 있다.


In [ ]:
# 1. BM25는 원문 질문을 받아 list[doc_id]를 직접 반환한다.
bm25_results = {}
for _, row in queries_df.iterrows():
    qid = row["query_id"]
    bm25_results[qid] = bm25_search(row["query_text"], top_k=5)

# 2. 일반 Dense는 원문 질문을 검색하고 list[Document]를 list[doc_id]로 바꾼다.
dense_results = {}
for _, row in queries_df.iterrows():
    qid = row["query_id"]
    # 일반 Dense는 질문 원문을 바로 임베딩한다.
    docs = vector_store.similarity_search(row["query_text"], k=5)
    dense_results[qid] = [doc.metadata["doc_id"] for doc in docs]

# 3. HyDE는 가상 답변을 검색하고 같은 list[doc_id] 계약으로 맞춘다.
hyde_results = {}
for _, row in hyde_pseudo_df.iterrows():
    qid = row["query_id"]
    # HyDE는 질문 대신 가상 문서를 같은 벡터 스토어에 전달한다.
    docs = vector_store.similarity_search(row["pseudo_answer"], k=5)
    hyde_results[qid] = [doc.metadata["doc_id"] for doc in docs]


## BM25 결과 표본 확인

질의별 BM25 상위 문서 ID를 먼저 확인한다. 각 value는 BM25 점수가 아니라 높은 점수부터 나열한 문서 ID 목록이며, 이어지는 Dense·HyDE 결과와 같은 `query_id`로 비교한다.

### 코드 해석 순서

1. HyDE 평가의 희소 검색 기준선 딕셔너리를 확인한다.

### 결과 해석

- 실행하면 각 `query_id`에 대해 BM25 점수가 양수인 문서 ID가 높은 점수 순서로 나타난다. 후보가 다섯 개보다 적으면 결과도 짧아질 수 있다.
- BM25 결과는 표현이 정확히 겹치는 문서가 앞서며 Dense·HyDE와 다른 오류 패턴을 제공한다.


In [ ]:
# 각 value는 점수 자체가 아니라 최대 다섯 개 문서 ID의 순서이다.
# Q1과 몇 개의 질의를 표본으로 정답 ID가 몇 위에 있는지 읽는다.
bm25_results


## 일반 Dense 결과 표본 확인

원문 질문을 바로 임베딩한 질의별 문서 ID 순서를 확인한다. 이 결과는 같은 임베딩 모델과 Pinecone index를 사용하면서 입력만 가상 답변으로 바꾸는 HyDE의 직접 기준선이다.

### 코드 해석 순서

1. 질문 임베딩 기반 Dense 결과의 문서 ID 순서를 확인한다.

### 결과 해석

- 저장 출력에서 Q1 Dense 결과는 D1, D12, D8, D2, D23 순서이다.
- 원문 질문 Dense와 HyDE의 차이는 임베딩 모델이 아니라 입력 텍스트 확장 여부에서 나온다.


In [ ]:
# Q1의 value는 Pinecone 코사인 순위 상위 다섯 ID이다.
# BM25와 겹치는 D1·D2 외에 의미 검색이 추가한 후보를 관찰한다.
dense_results


## HyDE 결과 표본 확인

가상 답변을 임베딩해 검색한 질의별 문서 ID 순서를 확인한다. 가상 답변 자체를 인용하지 않고, 이 ID로 찾아온 실제 `Document.page_content`와 메타데이터를 RAG의 근거 후보로 사용한다.

### 코드 해석 순서

1. 가상 문서 임베딩 기반 검색 결과의 순서를 확인한다.

### 결과 해석

- HyDE 외부 경로를 실행하면 생성된 가상 답변에 따른 질의별 문서 ID 목록이 표시된다.
- 생성 모델이나 temperature가 달라지면 가상 답변과 검색 순위도 달라질 수 있으므로 실행 결과로 평가한다.


In [ ]:
# 각 value는 생성 문장을 검색한 실제 corpus 문서 ID 다섯 개이다.
# Q1에서 일반 Dense와 비교해 정답 문서가 추가되거나 순위가 이동했는지 본다.
hyde_results


## BM25·Dense·HyDE 지표 비교

세 결과를 같은 qrels에 평가해 P@5, R@5, MRR와 MAP를 표로 만든다. HyDE가 회수율이나 상위 순서를 개선하는지 확인하고, LLM 호출 수와 지연이 그 개선을 정당화하는지도 함께 판단한다.

다음 번호의 Cohere Rerank는 HyDE 결과를 이어받지 않는 별도 심화 분기이다. HyDE가 질의 표현을 바꾸는 반면 Cohere Rerank는 자체 후보 집합의 상위 순서를 조정한다.

### 코드 해석 순서

1. 같은 evaluate_all()로 세 결과 딕셔너리의 평균 지표를 계산한다.
2. 네 지표를 같은 행에 배치해 방법별 차이와 추가 호출 비용을 함께 판단한다.

### 결과 해석

- 전체 외부 경로를 실행하면 수정된 AP@5 정의에 따른 세 검색 방식 비교표가 표시된다.
- HyDE가 개선되지 않으면 생성 프롬프트, 가상 문서 길이, 질의 유형과 추가 비용을 함께 재검토해야 한다.


In [ ]:
# 1. 같은 evaluate_all()로 세 결과 딕셔너리의 평균 지표를 계산한다.
# dense_metrics는 원문 질문 Dense이고 hyde_metrics는 가상 답변 Dense의 평균이다.
bm25_metrics = evaluate_all(bm25_results, queries_df)
dense_metrics = evaluate_all(dense_results, queries_df)
hyde_metrics = evaluate_all(hyde_results, queries_df)

# 2. 네 지표를 같은 행에 배치해 방법별 차이와 추가 호출 비용을 함께 판단한다.
metrics_df = pd.DataFrame({
    'Metric': ['P@5', 'R@5', 'MRR', 'MAP'],
    'BM25': [bm25_metrics['P@k'], bm25_metrics['R@k'], bm25_metrics['MRR'], bm25_metrics['MAP']],
    'Dense': [dense_metrics['P@k'], dense_metrics['R@k'], dense_metrics['MRR'], dense_metrics['MAP']],
    'HyDE': [hyde_metrics['P@k'], hyde_metrics['R@k'], hyde_metrics['MRR'], hyde_metrics['MAP']]
})
metrics_df
